In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/validation/final_validation.csv
/kaggle/input/test-split/split1_test.csv
/kaggle/input/test-split/split2_test.csv
/kaggle/input/scienceqa-image-url/validation.csv
/kaggle/input/scienceqa-image-url/train.csv
/kaggle/input/scienceqa-image-url/test.csv
/kaggle/input/with-scene-graph/split2_validation.csv
/kaggle/input/with-scene-graph/split1_validation.csv


# Dataset-1

In [2]:
from datasets import load_dataset

In [3]:
original = load_dataset("/kaggle/input/scienceqa-image-url")
original

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url'],
        num_rows: 6218
    })
    validation: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url'],
        num_rows: 2017
    })
    test: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url'],
        num_rows: 2017
    })
})

In [4]:
ScienceQA = load_dataset("/kaggle/input/test-split")

Generating test split: 0 examples [00:00, ? examples/s]

In [5]:
ScienceQA

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url', 'KG'],
        num_rows: 2017
    })
})

In [6]:
from datasets import DatasetDict

# Create a new DatasetDict with only the rows where "KG" is None
noneQA = DatasetDict({
    "test": ScienceQA["test"].filter(lambda example: example["KG"] is None)
})


Filter:   0%|          | 0/2017 [00:00<?, ? examples/s]

# KG

In [ ]:
# from google import genai

# client = genai.Client(api_key="")

In [8]:
# from google.genai import types

In [9]:
def generate_scene_graph(row, idx):
    # Download the image
    try:
        image_response = requests.get(row["image_url"])
    except Exception as e:
        print(f"Error downloading image at index {idx}: {e}")
        row['KG'] = None
        return row

    if image_response.status_code != 200:
        print(f"Failed to download image at index {idx}. Status code: {image_response.status_code}")
        row['KG'] = None
        return row

    image_bytes = image_response.content

    # Build the text prompt for the model
    prompt_text = (
        "Question: " + str(row["question"]) +
        " Choices: " + str(row["choices"]) +
        " For the provided image and its associated question, generate a scene graph in JSON format that includes the following: "
        "1. Objects that are relevant to answering the question. "
        "2. Object attributes that are relevant to answering the question. "
        "3. Object relationships that are relevant to answering the question. "
        "Scene Graph:"
    )

    max_retries = 3
    attempt = 0
    response = None

    while attempt < max_retries:
        try:
            response = client.models.generate_content(
                contents=[
                    types.Part.from_text(prompt_text),
                    types.Part.from_bytes(image_bytes, mime_type="image/png")
                ],
                model="gemini-2.0-flash"
            )
            # Successfully obtained response, exit retry loop
            break
        except Exception as e:
            error_message = str(e)
            # Check if error is a timeout or resource exhaustion error (429)
            if "408" in error_message or "RESOURCE_EXHAUSTED" in error_message or "429" in error_message:
                print(f"Index {idx}: {error_message} on attempt {attempt+1}. Retrying in 10 seconds...")
                time.sleep(15)
                attempt += 1
            else:
                print(f"Index {idx}: Unexpected error: {error_message}. Skipping this row.")
                attempt = max_retries  # exit loop; will mark as None
    # If response is still None after retries, mark KG as None
    if response is None:
        print(f"Index {idx}: Max retries reached or error occurred. Setting KG to None.")
        row['KG'] = None
    else:
        try:
            scene_graph = response.candidates[0].content.parts
            row['KG'] = scene_graph[0].text
        except Exception as e:
            print(f"Index {idx}: Error processing response: {e}. Setting KG to None.")
            row['KG'] = None


    time.sleep(5)
    return row

# scene graph generation

## split val set

In [11]:
import time

In [12]:
import requests

In [13]:
from datasets import DatasetDict


In [14]:
# validation_ds = ScienceQA['validation']
# split1 = validation_ds.select(range(1200))
# split2 = validation_ds.select(range(1200, len(validation_ds)))


In [ ]:
from google import genai
from google.genai import types
client = genai.Client(api_key="########################") # change with own key

In [16]:
noneQA

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url', 'KG'],
        num_rows: 34
    })
})

In [17]:
noneQA = noneQA.map(generate_scene_graph,with_indices=True, load_from_cache_file=False)

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Index 0: Unexpected error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}. Skipping this row.
Index 0: Max retries reached or error occurred. Setting KG to None.
Index 6: Unexpected error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}. Skipping this row.
Index 6: Max retries reached or error occurred. Setting KG to None.
Index 7: Unexpected error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}. Skipping this row.
Index 7: Max retries reached or error occurred. Setting KG to None.
Index 13: Unexpected error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}. Skipping this row.
Index 13: Max retries reached or error occurred. Setting KG to None.
Index 14: Unexpected error: 503 UNAVAILABLE. {'error': {'code'

In [18]:
from datasets import concatenate_datasets

# Get the rows where "KG" is not None
qa_dataset = ScienceQA["test"].filter(lambda example: example["KG"] is not None)

# After any manipulation on noneQA (if applicable), mix them together:
combined_dataset = DatasetDict({
    "test": concatenate_datasets([qa_dataset, noneQA["test"]])
})

Filter:   0%|          | 0/2017 [00:00<?, ? examples/s]

In [19]:
combined_dataset

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution', 'image_url', 'KG'],
        num_rows: 2017
    })
})

In [20]:
import pandas as pd

# Convert the "validation" split to a pandas DataFrame
df = combined_dataset["test"].to_pandas()

# Save the DataFrame as a CSV file
df.to_csv('final_test.csv', index=False)


In [21]:
# test_ds = ScienceQA['test']
# test_split1 = test_ds.select(range(1200))
# test_split2 = test_ds.select(range(1200, len(test_ds)))

In [22]:
# test_split1 = test_split1.map(generate_scene_graph, with_indices=True, load_from_cache_file=False)
# test_split1.to_csv('split1_test.csv')

In [23]:
# test_split2 = test_split2.map(generate_scene_graph, with_indices=True, load_from_cache_file=False)

# # Save the processed dataset to CSV for later use
# test_split2.to_csv('split2_test.csv')

In [24]:

# split1 = split1.map(generate_scene_graph,with_indices=True, load_from_cache_file=False) # change it to split2
# split1.to_csv('split1_validation.csv') # change this as well 


In [25]:
# split2 = split2.map(generate_scene_graph, with_indices=True, load_from_cache_file=False)

# # Save the processed dataset to CSV for later use
# split2.to_csv('split2_validation.csv')